In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path(r"E:\Projet_Alzheimer")
AUDIT = (
    ROOT / "data" / "05_Manifestes"
    / "audit_20261023_133400_145613"
)

# Recharger les résultats enregistrés.
candidats = pd.read_csv(
    AUDIT / "candidats_provisoires.csv",
    dtype={"PTID": "string"},
)

paires = pd.read_csv(
    AUDIT / "rapprochements_irm_pet_provisoires.csv",
    dtype={"PTID": "string"},
    parse_dates=["date_pet", "date_irm"],
)

# Mesurer le suivi IRM pour chaque candidat.
suivi = paires.groupby("PTID").agg(
    premiere_irm=("date_irm", "min"),
    derniere_irm=("date_irm", "max"),
    nombre_dates_irm=("date_irm", "nunique"),
    nombre_dates_pet=("date_pet", "nunique"),
)

suivi["duree_suivi_jours"] = (
    suivi["derniere_irm"] - suivi["premiere_irm"]
).dt.days

print("Participants rechargés :", len(candidats))

bilan = pd.DataFrame([
    {
        "duree_minimale_suivi_IRM": libelle,
        "participants": int(
            (suivi["duree_suivi_jours"] >= jours).sum()
        ),
    }
    for libelle, jours in [
        ("Sans minimum supplémentaire", 0),
        ("180 jours", 180),
        ("365 jours", 365),
        ("730 jours", 730),
    ]
])

display(bilan)

# Repérer les IRM utilisées pour plusieurs dates PET.
reutilisations = (
    paires.groupby(["PTID", "date_irm"])["date_pet"].nunique()
)

print(
    "Dates IRM associées à plusieurs dates PET :",
    int((reutilisations > 1).sum()),
)

In [ ]:
# Recharger les diagnostics : ce notebook reste indépendant.
TABLES = ROOT / "data" / "02_Tables_originales"

fichiers_dx = list(TABLES.rglob("All_Subjects_DXSUM_*.csv"))
if len(fichiers_dx) != 1:
    raise ValueError("Un seul fichier DXSUM est attendu.")

dx = pd.read_csv(
    fichiers_dx[0],
    dtype="string",
    encoding="utf-8-sig",
)

dx["PTID"] = dx["PTID"].str.strip().str.upper()
dx["DIAGNOSIS"] = dx["DIAGNOSIS"].str.strip()
dx["date_dx"] = pd.to_datetime(
    dx["EXAMDATE"].str.strip(),
    format="%Y-%m-%d",
    errors="coerce",
)

dx_valides = dx.loc[
    dx["PTID"].isin(candidats["PTID"])
    & dx["DIAGNOSIS"].isin(["1", "2", "3"])
    & dx["date_dx"].notna()
].copy()

# Écarter provisoirement les visites aux codes contradictoires.
codes_par_date = (
    dx_valides.groupby(["PTID", "date_dx"])["DIAGNOSIS"].nunique()
)
conflits = codes_par_date[codes_par_date > 1].index

index_dx = pd.MultiIndex.from_frame(
    dx_valides[["PTID", "date_dx"]]
)

dx_valides = (
    dx_valides.loc[
        ~index_dx.isin(conflits),
        ["PTID", "date_dx", "DIAGNOSIS"],
    ]
    .drop_duplicates()
)

visites = paires[["PTID", "date_irm"]].drop_duplicates()

# Audit de proximité uniquement : diagnostic avant ou après l'IRM.
audit_diagnostics = pd.merge_asof(
    visites.sort_values("date_irm"),
    dx_valides.sort_values("date_dx"),
    by="PTID",
    left_on="date_irm",
    right_on="date_dx",
    direction="nearest",
)

audit_diagnostics["ecart_jours"] = (
    audit_diagnostics["date_irm"] - audit_diagnostics["date_dx"]
).dt.days.abs()

resultats = []

for seuil in [30, 90]:
    proches = audit_diagnostics.loc[
        audit_diagnostics["ecart_jours"] <= seuil
    ]

    comptes = proches.groupby("PTID").agg(
        dates_irm=("date_irm", "nunique"),
        dates_dx=("date_dx", "nunique"),
        premiere_irm=("date_irm", "min"),
        derniere_irm=("date_irm", "max"),
    )

    duree = (
        comptes["derniere_irm"] - comptes["premiere_irm"]
    ).dt.days

    admissibles = (
        (comptes["dates_irm"] >= 2)
        & (comptes["dates_dx"] >= 2)
    )

    resultats.append({
        "ecart_IRM_diagnostic_max_jours": seuil,
        "participants_avec_2_dates": int(admissibles.sum()),
        "dont_suivi_IRM_365_jours": int(
            (admissibles & (duree >= 365)).sum()
        ),
        "dont_suivi_IRM_730_jours": int(
            (admissibles & (duree >= 730)).sum()
        ),
    })

display(pd.DataFrame(resultats))

In [ ]:
# Visites IRM avec un diagnostic proche.
visites_proches = audit_diagnostics.loc[
    audit_diagnostics["ecart_jours"] <= 30,
    ["PTID", "date_irm", "date_dx", "DIAGNOSIS"],
].copy()

# Appliquer les critères aux mêmes visites IRM–PET–diagnostic.
visites_pilote = paires.merge(
    visites_proches,
    on=["PTID", "date_irm"],
    how="inner",
    validate="many_to_one",
)

bilan_pilote = visites_pilote.groupby("PTID").agg(
    dates_irm=("date_irm", "nunique"),
    dates_pet=("date_pet", "nunique"),
    dates_diagnostic=("date_dx", "nunique"),
    premiere_irm=("date_irm", "min"),
    derniere_irm=("date_irm", "max"),
)

bilan_pilote["suivi_jours"] = (
    bilan_pilote["derniere_irm"] - bilan_pilote["premiere_irm"]
).dt.days

retenus = bilan_pilote.loc[
    (bilan_pilote["dates_irm"] >= 2)
    & (bilan_pilote["dates_pet"] >= 2)
    & (bilan_pilote["dates_diagnostic"] >= 2)
    & (bilan_pilote["suivi_jours"] >= 365)
].copy()

visites_pilote = visites_pilote.loc[
    visites_pilote["PTID"].isin(retenus.index)
].copy()

print("Participants satisfaisant les critères combinés :", len(retenus))

# Distribution du code diagnostique à la première IRM retenue.
premieres_visites = (
    visites_pilote.sort_values(["PTID", "date_irm"])
    .drop_duplicates("PTID", keep="first")
)

display(
    premieres_visites["DIAGNOSIS"]
    .value_counts()
    .sort_index()
    .rename_axis("code_diagnostique_initial")
    .reset_index(name="participants")
)